# BLaVe-CoT — Demo train LoRA BLIP-2 trên Colab

Notebook này train **bản demo nhỏ** (200 mẫu, 2 epoch) chỉ để **kiểm chứng pipeline chạy thông**.
Không nhằm tạo mô hình tốt. Bản đầy đủ chạy trên GPU mạnh với `PROFILE='full'`.

**Yêu cầu:** Runtime → Change runtime type → GPU (T4).

## 1. Cài thư viện

In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.0 bitsandbytes==0.43.0 sentencepiece
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Lấy code dự án
Tải các file .py (config, model, train, prepare_data) lên Colab, hoặc clone từ git repo của bạn.

In [ ]:
# Cách 1: upload thủ công các file .py qua panel Files bên trái
# Cách 2: nếu đã đẩy lên git:
# !git clone <repo-cua-ban> && cd <repo>
import os; print('Các file hiện có:', [f for f in os.listdir('.') if f.endswith('.py')])

## 3. Tải dữ liệu VizWiz (chỉ Annotations + một phần ảnh)
Tải nguyên train.zip (~10.5GB) lên Colab miễn phí là không thực tế.
Cho demo, ta tải Annotations (nhẹ) và chỉ những ảnh cần cho 200 mẫu đầu.

In [ ]:
import os
os.makedirs('data', exist_ok=True)
# Tải annotations (nhẹ)
!wget -q https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/Annotations.zip -O data/Ann.zip
!unzip -q -o data/Ann.zip -d data/vizwiz/
print('Đã tải annotations.')

In [ ]:
# Chuyển 200 mẫu đầu sang train_converted.json, đồng thời xuất danh sách ảnh cần
!python prepare_data.py --raw data/vizwiz/Annotations/train.json --out data/train_converted.json --max_samples 200 --list_images data/needed.txt
print(open('data/needed.txt').read()[:300])

In [ ]:
# Tải train.zip rồi CHỈ giải nén những ảnh cần (tiết kiệm dung lượng).
# Lưu ý: bước tải vẫn ~10.5GB; nếu Colab hết dung lượng, cân nhắc mount Google Drive.
!wget -q https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip -O data/train.zip
needed = set(open('data/needed.txt').read().split())
import zipfile, os
os.makedirs('data/vizwiz/train', exist_ok=True)
with zipfile.ZipFile('data/train.zip') as z:
    members = [n for n in z.namelist() if os.path.basename(n) in needed]
    for m in members:
        data = z.read(m)
        open(os.path.join('data/vizwiz/train', os.path.basename(m)), 'wb').write(data)
print(f'Đã giải nén {len(members)} ảnh cần cho demo.')

## 4. Đảm bảo config đang ở chế độ demo
Mở `config.py`, đặt `PROFILE = 'demo'`. Kiểm tra:

In [ ]:
!python config.py

## 5. Train demo
Nếu báo lỗi 'target modules not found' → chạy `!python inspect_model.py` rồi sửa target_modules trong config.

In [ ]:
!python train.py

## 6. Thử mô hình vừa train
So sánh đáp án trước/sau fine-tune trên một ảnh val.

In [ ]:
import json
sample = json.load(open('data/train.json'))[0]
img = f"data/vizwiz/train/{sample['image']}"
!python infer.py --adapter ./blip2_lora_demo/final --image "{img}" --question "{sample['question']}" --compare